# 🧠 Autoencoders y Embeddings — Notebook Completo
> **Curso:** Machine Learning / Business Intelligence
> **Objetivo:** Entender representaciones aprendidas, compresión semántica y espacio latente

> 💡 **Tip:** Activa la GPU en Colab: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`

---
## ¿Qué aprenderás?
1. Qué son los **embeddings** y por qué son la base del AI moderno
2. Arquitectura de un **autoencoder** (encoder → espacio latente → decoder)
3. Entrenar y evaluar con **pérdida de reconstrucción**
4. Explorar el **espacio latente**: visualización PCA/t-SNE
5. **Interpolación** entre imágenes en el espacio latente
6. **Aritmética de vectores**: analogías semánticas
7. **Autoencoder denoising**: aprender a eliminar ruido
8. **Detección de anomalías** con error de reconstrucción

---
### ¿Por qué importan los embeddings?

| Objeto | Espacio original | Embedding | Ejemplo |
|---|---|---|---|
| Imagen 28×28 | 784 pixeles | 8 dimensiones | Dígito → coordenada en espacio semántico |
| Palabra | 50.000 tokens | 128-1536 dim. | 'King' - 'Man' + 'Woman' ≈ 'Queen' |
| Cliente | 200 variables | 16 dimensiones | Perfil comprimido para recomendaciones |
| Producto | Catálogo 100k | 32 dimensiones | Similitud semántica sin búsqueda exhaustiva |

## 1. Instalación y Configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.datasets import fashion_mnist

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import cosine, euclidean

plt.rcParams.update({
    'figure.facecolor': '#18181b', 'axes.facecolor': '#27272a',
    'axes.edgecolor': '#3f3f46', 'text.color': '#e4e4e7',
    'axes.labelcolor': '#a1a1aa', 'xtick.color': '#71717a',
    'ytick.color': '#71717a', 'grid.color': '#3f3f46', 'grid.alpha': 0.4,
})

PALETTE = ['#10b981','#3b82f6','#f59e0b','#8b5cf6','#ef4444',
           '#ec4899','#06b6d4','#f97316','#22c55e','#64748b']

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')
np.random.seed(42)
tf.random.set_seed(42)

---
## 2. Dataset — Fashion-MNIST

**Fashion-MNIST** reemplaza el clásico MNIST con prendas de ropa. Más difícil y más relevante para el mundo real.

| Clase | Prenda | Clase | Prenda |
|---|---|---|---|
| 0 | T-shirt/top | 5 | Sandal |
| 1 | Trouser | 6 | Shirt |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Bag |
| 4 | Coat | 9 | Ankle boot |

In [ ]:
(X_train_raw, y_train), (X_test_raw, y_test) = fashion_mnist.load_data()

CLASS_NAMES = ['T-Shirt','Pantalón','Pullover','Vestido','Abrigo',
               'Sandalia','Camisa','Zapatilla','Bolso','Botín']

print(f'Train: {X_train_raw.shape}  |  Test: {X_test_raw.shape}')
print(f'Clases: {np.unique(y_train)}')
print(f'Distribución clases: {np.bincount(y_train)} (balanceado)')

In [ ]:
# Visualizar muestras por clase
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
fig.suptitle('Fashion-MNIST — 2 muestras por clase', fontsize=12)
for cls in range(10):
    idxs = np.where(y_train == cls)[0][:2]
    for row, idx in enumerate(idxs):
        ax = axes[row, cls]
        ax.imshow(X_train_raw[idx], cmap='gray', vmin=0, vmax=255)
        if row == 0: ax.set_title(CLASS_NAMES[cls], fontsize=8)
        ax.axis('off')
plt.tight_layout(); plt.show()

---
## 3. Preprocesamiento

In [ ]:
# Normalizar a [0,1] y aplanar a vectores de 784
X_train = X_train_raw.astype('float32') / 255.0
X_test  = X_test_raw.astype('float32')  / 255.0

X_train_flat = X_train.reshape(-1, 784)
X_test_flat  = X_test.reshape(-1, 784)

print(f'Shape después de preprocesamiento: {X_train_flat.shape}')
print(f'Rango de valores: [{X_train_flat.min():.2f}, {X_train_flat.max():.2f}]')
print()
print('Diferencia vs AR/Clustering:')
print('  AR:           transacciones → matriz binaria (binariza)')
print('  Clustering:   tabular → escalar (Z-score o Min-Max)')
print('  Autoencoder:  imágenes → normalizar a [0,1], sin binarizar')

---
## 4. Arquitectura del Autoencoder

```
Input (784)  →  Encoder  →  Latent Space (8)  →  Decoder  →  Output (784)
              784→256→64→8                    8→64→256→784
```

**Cuello de botella (bottleneck):** El autoencoder **debe aprender a comprimir** la imagen en solo 8 números. Para reconstruirla bien, esos 8 números deben capturar las características más importantes.

> 🔑 **Insight:** Nadie le dijo a la red qué características extraer. El espacio latente emerge solo del proceso de comprimir y descomprimir.

In [ ]:
LATENT_DIM = 8   # ← dimensión del espacio latente (prueba con 2, 4, 8, 16, 32)

# Encoder
encoder_input = keras.Input(shape=(784,), name='encoder_input')
x = layers.Dense(256, activation='relu', name='enc_256')(encoder_input)
x = layers.Dense(64,  activation='relu', name='enc_64')(x)
latent = layers.Dense(LATENT_DIM, activation='linear', name='latent')(x)
encoder = Model(encoder_input, latent, name='encoder')

# Decoder
decoder_input = keras.Input(shape=(LATENT_DIM,), name='decoder_input')
x = layers.Dense(64,  activation='relu', name='dec_64')(decoder_input)
x = layers.Dense(256, activation='relu', name='dec_256')(x)
decoder_output = layers.Dense(784, activation='sigmoid', name='output')(x)
decoder = Model(decoder_input, decoder_output, name='decoder')

# Autoencoder completo
ae_input  = keras.Input(shape=(784,))
ae_output = decoder(encoder(ae_input))
autoencoder = Model(ae_input, ae_output, name='autoencoder')
autoencoder.compile(optimizer='adam', loss='mse')

# Resumen
print('ENCODER:')
encoder.summary()
print()
print('DECODER:')
decoder.summary()
print()
print(f'Ratio de compresión: 784 → {LATENT_DIM}  ({784/LATENT_DIM:.0f}:1)')

---
## 5. Entrenamiento

In [ ]:
EPOCHS     = 30    # ← prueba con 10, 30, 50
BATCH_SIZE = 256

history = autoencoder.fit(
    X_train_flat, X_train_flat,          # ← input y output son la misma imagen
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_flat, X_test_flat),
    verbose=1,
)

# Curvas de pérdida
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'],     color='#10b981', linewidth=2, label='Train Loss (MSE)')
ax.plot(history.history['val_loss'], color='#3b82f6', linewidth=2, linestyle='--', label='Val Loss (MSE)')
ax.set_xlabel('Época')
ax.set_ylabel('MSE (error de reconstrucción)')
ax.set_title('Curvas de Pérdida del Autoencoder')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

final_train = history.history['loss'][-1]
final_val   = history.history['val_loss'][-1]
print(f'Train MSE final: {final_train:.6f}')
print(f'Val   MSE final: {final_val:.6f}')
if final_val > final_train * 1.5:
    print('⚠️  Gap grande → posible overfitting, considera reducir epochs o añadir dropout')
else:
    print('✅ Train y Val convergen bien — buena generalización')

---
## 6. Calidad de Reconstrucción

In [ ]:
# Reconstruir imágenes de test
X_reconstructed = autoencoder.predict(X_test_flat, verbose=0)

# Mostrar original vs reconstruida para cada clase
n_show = 10
fig, axes = plt.subplots(3, n_show, figsize=(16, 5))
fig.suptitle('Reconstrucción por Clase: Original | Reconstruida | Error', fontsize=11)

sample_idxs = [np.where(y_test == c)[0][0] for c in range(n_show)]
for col, idx in enumerate(sample_idxs):
    orig  = X_test_flat[idx].reshape(28, 28)
    recon = X_reconstructed[idx].reshape(28, 28)
    err   = np.abs(orig - recon)
    axes[0, col].imshow(orig,  cmap='gray', vmin=0, vmax=1)
    axes[1, col].imshow(recon, cmap='gray', vmin=0, vmax=1)
    axes[2, col].imshow(err,   cmap='Reds', vmin=0, vmax=0.3)
    axes[0, col].set_title(CLASS_NAMES[y_test[idx]], fontsize=7)
    for row in range(3): axes[row, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=8, color='#a1a1aa')
axes[1, 0].set_ylabel('Reconstruida', fontsize=8, color='#a1a1aa')
axes[2, 0].set_ylabel('Error |orig-recon|', fontsize=8, color='#a1a1aa')
for ax_row in [axes[0,0], axes[1,0], axes[2,0]]:
    ax_row.axis('on'); ax_row.set_xticks([]); ax_row.set_yticks([])

plt.tight_layout(); plt.show()

In [ ]:
# Error de reconstrucción por clase
recon_errors = np.mean((X_test_flat - X_reconstructed)**2, axis=1)

fig, ax = plt.subplots(figsize=(11, 4))
for cls in range(10):
    mask  = y_test == cls
    errors_cls = recon_errors[mask]
    ax.boxplot(errors_cls, positions=[cls], widths=0.6,
               patch_artist=True,
               boxprops=dict(facecolor=PALETTE[cls], alpha=0.7),
               medianprops=dict(color='white', linewidth=2),
               whiskerprops=dict(color='#a1a1aa'),
               capprops=dict(color='#a1a1aa'),
               flierprops=dict(marker='.', color=PALETTE[cls], alpha=0.3))

ax.set_xticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('MSE de reconstrucción')
ax.set_title('Error de Reconstrucción por Clase\n(mayor error = clase más difícil de comprimir)')
ax.grid(axis='y')
plt.tight_layout(); plt.show()

---
## 7. Espacio Latente — Visualización

Vamos a comprimir 784 dimensiones a 8 (con el encoder) y luego a 2 (con PCA o t-SNE) para visualizar cómo organiza el modelo las prendas en el espacio latente.

In [ ]:
# Obtener embeddings del conjunto de test
N_VIZ = 5000  # submuestra para velocidad
idx_viz = np.random.choice(len(X_test_flat), N_VIZ, replace=False)
X_viz   = X_test_flat[idx_viz]
y_viz   = y_test[idx_viz]

embeddings = encoder.predict(X_viz, verbose=0)   # shape: (N_VIZ, LATENT_DIM)
print(f'Embeddings shape: {embeddings.shape}')
print(f'Cada imagen → vector de {LATENT_DIM} dimensiones')

# PCA 2D del espacio latente
pca2 = PCA(n_components=2, random_state=42)
emb_2d_pca = pca2.fit_transform(embeddings)
print(f'PCA varianza explicada: {pca2.explained_variance_ratio_.sum():.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Espacio Latente del Autoencoder — Organización Semántica sin Supervisión', fontsize=12)

# ── PCA ───────────────────────────────────────────────────────────
for cls in range(10):
    mask = y_viz == cls
    axes[0].scatter(emb_2d_pca[mask, 0], emb_2d_pca[mask, 1],
                    c=PALETTE[cls], s=5, alpha=0.6, label=CLASS_NAMES[cls])
axes[0].set_title('PCA 2D del Espacio Latente (8→2)')
axes[0].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
axes[0].legend(fontsize=7, ncol=2, markerscale=3)

# ── t-SNE (más fiel a la estructura local) ────────────────────────
print('Calculando t-SNE (puede tardar 1-2 minutos)...')
tsne = TSNE(n_components=2, perplexity=30, n_iter=500, random_state=42, verbose=0)
emb_2d_tsne = tsne.fit_transform(embeddings)

for cls in range(10):
    mask = y_viz == cls
    axes[1].scatter(emb_2d_tsne[mask, 0], emb_2d_tsne[mask, 1],
                    c=PALETTE[cls], s=5, alpha=0.6, label=CLASS_NAMES[cls])
axes[1].set_title('t-SNE 2D del Espacio Latente (8→2)')
axes[1].legend(fontsize=7, ncol=2, markerscale=3)

plt.tight_layout(); plt.show()
print('🔑 El autoencoder aprendió a separar clases sin haber visto las etiquetas en ningún momento.')

---
## 8. Aritmética de Vectores en el Espacio Latente

En un espacio latente bien organizado, las **operaciones aritméticas tienen sentido semántico**.

El ejemplo más famoso con word embeddings:
```
embedding('King') - embedding('Man') + embedding('Woman') ≈ embedding('Queen')
```

Con Fashion-MNIST podemos explorar:
```
Zapatilla - Sandalia + Vestido ≈ ?
```

In [ ]:
def class_centroid(cls_id):
    """Embedding promedio de una clase = centroide en el espacio latente."""
    mask = y_test == cls_id
    return encoder.predict(X_test_flat[mask], verbose=0).mean(axis=0)

centroids = {CLASS_NAMES[i]: class_centroid(i) for i in range(10)}

def vector_arithmetic(a_name, b_name, c_name):
    """Realiza A - B + C y busca la clase más cercana al resultado."""
    result = centroids[a_name] - centroids[b_name] + centroids[c_name]
    # Imagen resultante decodificada
    result_img = decoder.predict(result.reshape(1, -1), verbose=0)[0].reshape(28, 28)
    # Clase más cercana por distancia euclideana
    dists = {name: euclidean(result, emb) for name, emb in centroids.items()}
    nearest = sorted(dists.items(), key=lambda x: x[1])
    return result_img, nearest[:3]

# Operaciones
operations = [
    ('Zapatilla', 'Sandalia', 'Vestido'),
    ('Botín',     'Zapatilla', 'Bolso'),
    ('Abrigo',    'Pullover',  'T-Shirt'),
]

fig, axes = plt.subplots(len(operations), 5, figsize=(12, 7))
fig.suptitle('Aritmética de Vectores en el Espacio Latente\nA - B + C → ?', fontsize=12)

for row, (a, b, c) in enumerate(operations):
    result_img, nearest = vector_arithmetic(a, b, c)
    a_idx = np.where(y_test == CLASS_NAMES.index(a))[0][0]
    b_idx = np.where(y_test == CLASS_NAMES.index(b))[0][0]
    c_idx = np.where(y_test == CLASS_NAMES.index(c))[0][0]
    for col, (img, title, cmap) in enumerate([
        (X_test_flat[a_idx].reshape(28,28), a, 'gray'),
        (X_test_flat[b_idx].reshape(28,28), f'− {b}', 'gray'),
        (X_test_flat[c_idx].reshape(28,28), f'+ {c}', 'gray'),
        (result_img, '= ?', 'gray'),
        (np.zeros((28,28)), f'→ {nearest[0][0]}', 'gray'),
    ]):
        if col < 4:
            axes[row, col].imshow(img, cmap=cmap, vmin=0, vmax=1)
        else:
            axes[row, col].text(0.5, 0.5, f'Clase más\ncercana:\n{nearest[0][0]}\n(d={nearest[0][1]:.2f})',
                                ha='center', va='center', fontsize=8, color='#fbbf24',
                                transform=axes[row, col].transAxes)
        axes[row, col].set_title(title, fontsize=8)
        axes[row, col].axis('off')
    # Separadores visuales
    for pos, sym in [(1, '−'), (2, '+'), (3, '=')]:
        axes[row, pos].set_title(f'{sym} {operations[row][pos-1] if pos < 3 else "?"}', fontsize=8)

plt.tight_layout(); plt.show()

---
## 9. Interpolación entre Imágenes

In [ ]:
def interpolate(img_a_flat, img_b_flat, steps=10):
    """Interpolación lineal en el espacio latente entre dos imágenes."""
    z_a = encoder.predict(img_a_flat.reshape(1, -1), verbose=0)[0]
    z_b = encoder.predict(img_b_flat.reshape(1, -1), verbose=0)[0]
    alphas = np.linspace(0, 1, steps)
    interpolated = []
    for alpha in alphas:
        z_interp = (1 - alpha) * z_a + alpha * z_b
        img = decoder.predict(z_interp.reshape(1, -1), verbose=0)[0].reshape(28, 28)
        interpolated.append(img)
    return interpolated

# Pares interesantes para interpolar
pairs = [
    (7, 5),   # Zapatilla → Sandalia
    (3, 0),   # Vestido → T-Shirt
    (4, 2),   # Abrigo → Pullover
]

STEPS = 10
fig, axes = plt.subplots(len(pairs), STEPS, figsize=(16, 5))
fig.suptitle('Interpolación en el Espacio Latente — Morfosis entre Prendas', fontsize=12)

for row, (cls_a, cls_b) in enumerate(pairs):
    idx_a = np.where(y_test == cls_a)[0][0]
    idx_b = np.where(y_test == cls_b)[0][0]
    frames = interpolate(X_test_flat[idx_a], X_test_flat[idx_b], STEPS)
    for col, img in enumerate(frames):
        axes[row, col].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
        if col == 0:     axes[row, col].set_title(CLASS_NAMES[cls_a], fontsize=7, color='#10b981')
        if col == STEPS-1: axes[row, col].set_title(CLASS_NAMES[cls_b], fontsize=7, color='#3b82f6')

plt.tight_layout(); plt.show()
print('🔑 La transición suave demuestra que el espacio latente es continuo y semánticamente organizado.')

---
## 10. Autoencoder Denoising

Entrenamos el autoencoder con **imágenes con ruido como input** y **imágenes limpias como target**. La red aprende a filtrar el ruido.

In [ ]:
# Añadir ruido gaussiano
NOISE_FACTOR = 0.35
X_train_noisy = np.clip(X_train_flat + NOISE_FACTOR * np.random.randn(*X_train_flat.shape), 0, 1)
X_test_noisy  = np.clip(X_test_flat  + NOISE_FACTOR * np.random.randn(*X_test_flat.shape),  0, 1)

# Nuevo autoencoder denoising (misma arquitectura)
dae_input  = keras.Input(shape=(784,))
dae_enc    = layers.Dense(256, activation='relu')(dae_input)
dae_enc    = layers.Dense(64,  activation='relu')(dae_enc)
dae_latent = layers.Dense(LATENT_DIM, activation='linear')(dae_enc)
dae_dec    = layers.Dense(64,  activation='relu')(dae_latent)
dae_dec    = layers.Dense(256, activation='relu')(dae_dec)
dae_out    = layers.Dense(784, activation='sigmoid')(dae_dec)
dae = Model(dae_input, dae_out, name='denoising_autoencoder')
dae.compile(optimizer='adam', loss='mse')

# input=noisy, target=clean
dae.fit(X_train_noisy, X_train_flat,
        epochs=20, batch_size=256,
        validation_data=(X_test_noisy, X_test_flat),
        verbose=1)

In [ ]:
# Comparar: original | con ruido | reconstruida limpia
X_denoised = dae.predict(X_test_noisy, verbose=0)

n_show = 10
sample_idxs = [np.where(y_test == c)[0][0] for c in range(n_show)]

fig, axes = plt.subplots(3, n_show, figsize=(16, 5))
fig.suptitle('Autoencoder Denoising: Original | Con Ruido | Reconstruida', fontsize=11)

for col, idx in enumerate(sample_idxs):
    for row, (img_flat, title) in enumerate([
        (X_test_flat[idx],  CLASS_NAMES[y_test[idx]]),
        (X_test_noisy[idx], '+ ruido'),
        (X_denoised[idx],   'denoised'),
    ]):
        axes[row, col].imshow(img_flat.reshape(28,28), cmap='gray', vmin=0, vmax=1)
        if row == 0: axes[row, col].set_title(title, fontsize=7)
        axes[row, col].axis('off')

axes[0,0].set_ylabel('Original',  fontsize=8, color='#a1a1aa')
axes[1,0].set_ylabel('Con Ruido', fontsize=8, color='#a1a1aa')
axes[2,0].set_ylabel('Limpia',    fontsize=8, color='#a1a1aa')
for r in range(3): axes[r,0].axis('on'); axes[r,0].set_xticks([]); axes[r,0].set_yticks([])
plt.tight_layout(); plt.show()

---
## 11. Detección de Anomalías con Error de Reconstrucción

**Idea clave:** Si entrenas el autoencoder solo con datos normales, tendrá bajo error de reconstrucción en datos normales y **alto error en anomalías** (porque nunca aprendió a reconstruirlas).

Simulamos un escenario donde las prendas "normales" son 8 clases y las "anomalías" son 2 clases no vistas.

In [ ]:
# Simular: normal = clases 0-7, anomalías = clases 8 y 9 (Bolso y Botín)
NORMAL_CLASSES  = list(range(8))
ANOMALY_CLASSES = [8, 9]

# Entrenar un AE solo con clases normales
normal_mask_train = np.isin(y_train, NORMAL_CLASSES)
X_normal = X_train_flat[normal_mask_train]

ae_anomaly_input  = keras.Input(shape=(784,))
x = layers.Dense(256, activation='relu')(ae_anomaly_input)
x = layers.Dense(64,  activation='relu')(x)
x = layers.Dense(LATENT_DIM, activation='linear')(x)
x = layers.Dense(64,  activation='relu')(x)
x = layers.Dense(256, activation='relu')(x)
ae_anomaly_out = layers.Dense(784, activation='sigmoid')(x)
ae_anomaly = Model(ae_anomaly_input, ae_anomaly_out)
ae_anomaly.compile(optimizer='adam', loss='mse')
ae_anomaly.fit(X_normal, X_normal, epochs=15, batch_size=256, verbose=0)

# Calcular error de reconstrucción en test
X_recon_test = ae_anomaly.predict(X_test_flat, verbose=0)
recon_err    = np.mean((X_test_flat - X_recon_test)**2, axis=1)

# Distribución de errores: normales vs anomalías
is_anomaly = np.isin(y_test, ANOMALY_CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Detección de Anomalías — Error de Reconstrucción')

# Histograma
axes[0].hist(recon_err[~is_anomaly], bins=50, color='#10b981', alpha=0.7, label='Normal (clases 0-7)', density=True)
axes[0].hist(recon_err[is_anomaly],  bins=50, color='#ef4444', alpha=0.7, label='Anomalía (clases 8-9)', density=True)
threshold = np.percentile(recon_err[~is_anomaly], 95)
axes[0].axvline(threshold, color='#f59e0b', linewidth=2, linestyle='--', label=f'umbral p95 = {threshold:.4f}')
axes[0].set_xlabel('Error de Reconstrucción (MSE)')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución de Errores')
axes[0].legend(fontsize=8)

# Boxplot por clase
class_errors = [recon_err[y_test == c] for c in range(10)]
bp = axes[1].boxplot(class_errors,
                     patch_artist=True,
                     medianprops=dict(color='white', linewidth=2),
                     whiskerprops=dict(color='#a1a1aa'),
                     capprops=dict(color='#a1a1aa'),
                     flierprops=dict(marker='.', alpha=0.3))
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor('#ef4444' if i >= 8 else '#10b981')
    patch.set_alpha(0.7)
axes[1].set_xticks(range(1, 11))
axes[1].set_xticklabels(CLASS_NAMES, rotation=20, ha='right', fontsize=8)
axes[1].axhline(threshold, color='#f59e0b', linewidth=1.5, linestyle='--')
axes[1].set_title('Error por Clase (rojo = clase no vista durante entrenamiento)')
axes[1].grid(axis='y')

plt.tight_layout(); plt.show()

y_pred = (recon_err > threshold).astype(int)
y_true = is_anomaly.astype(int)
from sklearn.metrics import precision_score, recall_score, f1_score
print(f'Umbral = percentil 95 de errores normales: {threshold:.5f}')
print(f'Precisión : {precision_score(y_true, y_pred):.3f}')
print(f'Recall    : {recall_score(y_true, y_pred):.3f}')
print(f'F1-Score  : {f1_score(y_true, y_pred):.3f}')

---
## 12. Efecto de la Dimensión del Espacio Latente

¿Cómo afecta el tamaño del cuello de botella a la calidad de reconstrucción?

In [ ]:
def train_ae(latent_dim, epochs=20):
    inp = keras.Input(shape=(784,))
    x   = layers.Dense(256, activation='relu')(inp)
    x   = layers.Dense(64,  activation='relu')(x)
    z   = layers.Dense(latent_dim, activation='linear')(x)
    x   = layers.Dense(64,  activation='relu')(z)
    x   = layers.Dense(256, activation='relu')(x)
    out = layers.Dense(784, activation='sigmoid')(x)
    m   = Model(inp, out)
    m.compile(optimizer='adam', loss='mse')
    h   = m.fit(X_train_flat, X_train_flat, epochs=epochs,
                batch_size=256, validation_split=0.1, verbose=0)
    val_loss = min(h.history['val_loss'])
    return val_loss

latent_dims = [2, 4, 8, 16, 32, 64]
print('Entrenando autoencoders con distintas dimensiones latentes...')
val_losses = []
for ld in latent_dims:
    loss = train_ae(ld)
    val_losses.append(loss)
    print(f'  dim={ld:3d}  val_loss={loss:.6f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(latent_dims, val_losses, marker='o', color='#10b981', linewidth=2.5, markersize=8)
ax.fill_between(latent_dims, val_losses, alpha=0.15, color='#10b981')
ax.axvline(8, color='#f59e0b', linewidth=1.5, linestyle='--', label='LATENT_DIM=8 (este notebook)')
ax.set_xlabel('Dimensión del Espacio Latente')
ax.set_ylabel('MSE de Validación')
ax.set_title('Trade-off: Compresión vs Calidad de Reconstrucción')
ax.set_xscale('log', base=2)
ax.set_xticks(latent_dims); ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()
print('🔑 Con más dimensiones mejora la reconstrucción, pero el embedding pierde poder de generalización.')

---
## 13. Análisis de Dimensiones del Embedding

¿Qué "capturó" cada dimensión del espacio latente?

In [ ]:
# Obtener todos los embeddings del test
all_embeddings = encoder.predict(X_test_flat, verbose=0)  # (10000, LATENT_DIM)

# Para cada dimensión, variar su valor y ver qué cambia en la imagen decodificada
BASE_CLASS = 7  # Zapatilla
base_idx   = np.where(y_test == BASE_CLASS)[0][0]
base_emb   = encoder.predict(X_test_flat[[base_idx]], verbose=0)[0]

n_dims = min(LATENT_DIM, 8)
n_vals = 7
fig, axes = plt.subplots(n_dims, n_vals, figsize=(14, n_dims * 1.8))
fig.suptitle(f'Exploración de Dimensiones Latentes — imagen base: {CLASS_NAMES[BASE_CLASS]}', fontsize=11)

for dim in range(n_dims):
    dim_range = np.linspace(
        all_embeddings[:, dim].mean() - 2.5 * all_embeddings[:, dim].std(),
        all_embeddings[:, dim].mean() + 2.5 * all_embeddings[:, dim].std(),
        n_vals
    )
    for col, val in enumerate(dim_range):
        z_mod = base_emb.copy()
        z_mod[dim] = val
        img = decoder.predict(z_mod.reshape(1,-1), verbose=0)[0].reshape(28,28)
        axes[dim, col].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[dim, col].axis('off')
    axes[dim, 0].set_ylabel(f'D{dim}', fontsize=8, color='#a1a1aa')

for col, val in enumerate(np.linspace(-2.5, 2.5, n_vals)):
    axes[0, col].set_title(f'{val:.1f}σ', fontsize=7)

plt.tight_layout(); plt.show()
print('Cada fila controla un aspecto visual diferente: forma, grosor, orientación, textura...')

---
## 14. Ejercicios

### 👁️ Ejercicio 1 — Comportamiento sin entrenamiento
Crea un autoencoder nuevo (sin entrenar) y reconstruye 5 imágenes. ¿Qué ves? ¿Por qué el error de reconstrucción es alto? Esto ilustra que la red no tiene conocimiento antes de aprender.

### 📉 Ejercicio 2 — Efecto de la arquitectura
Agrega una capa Dropout(0.3) después de cada capa Dense del encoder. ¿Mejora o empeora la calidad de reconstrucción y la separación en el espacio latente?

### 🧩 Ejercicio 3 — Dimensión latente
Entrena autoencoders con `LATENT_DIM = 2` y `LATENT_DIM = 32`. Para `dim=2`, grafica directamente el espacio latente (sin PCA). ¿Se separan bien las clases? ¿Qué pasa con la calidad de reconstrucción?

### 🔍 Ejercicio 4 — Búsqueda por similitud
Dado un embedding cualquiera, usa `NearestNeighbors` de sklearn para encontrar las 5 imágenes más similares en el espacio latente. ¿Son de la misma clase? Esto simula un motor de búsqueda visual.

```python
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=6).fit(all_embeddings)
distances, indices = nn.kneighbors(query_embedding.reshape(1, -1))
```

### 💡 Ejercicio 5 — Conexión con LLMs
Los embeddings de texto en modelos como BERT o GPT tienen 768-4096 dimensiones. ¿Qué tipo de información crees que captura cada dimensión? ¿Cómo se diferencia del espacio latente de imágenes?